# 04 - Model Experiments and Evaluation
Compare TF-IDF and Sentence Transformer recommenders using offline ranking metrics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from recommender import (
    build_feature_column,
    build_sentence_embeddings,
    build_tfidf_model,
    evaluate_ranking,
    load_dataset,
    recommend_with_embeddings,
    recommend_with_tfidf,
)

sns.set_theme(style='whitegrid')

In [ ]:
df = build_feature_column(load_dataset('../data/kdrama_list_cleaned.csv'))
_, tfidf_matrix = build_tfidf_model(df['combined_text'], ngram_range=(1, 2))

def tfidf_fn(title, top_k):
    return recommend_with_tfidf(title, df, tfidf_matrix, top_k=top_k)

tfidf_metrics = evaluate_ranking(df, tfidf_fn, k=10, sample_size=200, random_state=42)
tfidf_metrics

In [ ]:
try:
    _, embeddings = build_sentence_embeddings(df['combined_text'].tolist())

    def sbert_fn(title, top_k):
        return recommend_with_embeddings(title, df, embeddings, top_k=top_k)

    sbert_metrics = evaluate_ranking(df, sbert_fn, k=10, sample_size=200, random_state=42)
except ImportError:
    sbert_metrics = {'precision@k': np.nan, 'recall@k': np.nan, 'mrr': np.nan}

comparison = pd.DataFrame([
    {'model': 'TF-IDF', **tfidf_metrics},
    {'model': 'SentenceTransformer', **sbert_metrics},
])
comparison

In [ ]:
sample_title = df.iloc[0]['Title']
sample_recs = recommend_with_tfidf(sample_title, df, tfidf_matrix, top_k=10)

plt.figure(figsize=(8, 5))
sns.histplot(sample_recs['similarity'], bins=10, kde=True, color='steelblue')
plt.title(f'Similarity Distribution for Recommendations of {sample_title}')
plt.xlabel('Cosine Similarity')
plt.ylabel('Count')
plt.show()

sample_recs[['Title', 'Genre', 'similarity']].head(10)

In [ ]:
dense = tfidf_matrix.toarray()
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(dense)

plot_df = pd.DataFrame({'x': coords[:, 0], 'y': coords[:, 1], 'score': df['Score']})
plt.figure(figsize=(8, 6))
sns.scatterplot(data=plot_df.sample(min(500, len(plot_df)), random_state=42), x='x', y='y', hue='score', palette='viridis', alpha=0.7)
plt.title('Drama Clusters (PCA Projection)')
plt.show()

# Optional t-SNE for deeper cluster visualization
# tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='pca')
# tsne_coords = tsne.fit_transform(dense[:800])